# DEP

In [2]:
!pip install transformers accelerate bitsandbytes
!pip install sentence-transformers
!pip install faiss-cpu
!pip install wikipedia
!pip install duckduckgo-search
!pip install requests
!pip install pandas
!pip install scikit-learn
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=8d1a6c55a2ee8ec96eeba7b56b1077b17947c7d7bc3f5df460381f64e5565acb
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 61.7 MB/s eta 0:00:00


In [3]:
import requests
import wikipedia
import pandas as pd
import numpy as np

from duckduckgo_search import DDGS

from sentence_transformers import SentenceTransformer

import faiss

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

import torch

from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [31]:
nltk.download('punkt_tab')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

True

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Models

In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [65]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Phi-3 loaded")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174c20>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174c20>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174c20>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d70>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7d2330174d70>
sys:1: ResourceWarning: Uncl

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Phi-3 loaded


# CONTEXT COLLECTOR

In [6]:
class ContextCollector:

    def collect_openfda(self, query):

        contexts = []

        try:

            url = (
                "https://api.fda.gov/drug/label.json"
                f"?search={query}"
                "&limit=5"
            )

            response = requests.get(url).json()

            for result in response.get("results", []):

                if "indications_and_usage" in result:

                    contexts.extend(
                        result["indications_and_usage"]
                    )

                if "warnings" in result:

                    contexts.extend(
                        result["warnings"]
                    )

        except Exception as e:
            print("FDA Error:", e)

        return contexts


    def collect_wikipedia(self, topics):

        data = []

        for topic in topics:

            try:

                summary = wikipedia.summary(
                    topic,
                    sentences=5
                )

                data.append(summary)

            except:
                pass

        return data


    def collect_duckduckgo(self, query):

        docs = []

        try:

            results = DDGS().text(
                query,
                max_results=10
            )

            for r in results:

                docs.append(
                    r.get("body", "")
                )

        except Exception as e:
            print("DuckDuckGo Error:", e)

        return docs

# Chunking

In [32]:
class Chunker:

    def chunk_text(
        self,
        documents,
        chunk_size=3
    ):

        chunks = []

        for doc in documents:

            sentences = sent_tokenize(doc)

            for i in range(
                0,
                len(sentences),
                chunk_size
            ):

                chunk = " ".join(
                    sentences[i:i+chunk_size]
                )

                chunks.append(chunk)

        return chunks

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

# VECTOR STORE

In [8]:
class VectorStore:

    def __init__(self):

        self.documents = []

        self.index = None


    def build_index(self, chunks):

        self.documents = chunks

        embeddings = embedding_model.encode(
            chunks,
            convert_to_numpy=True
        )

        dimension = embeddings.shape[1]

        self.index = faiss.IndexFlatL2(
            dimension
        )

        self.index.add(embeddings)

        print(
            f"Indexed {len(chunks)} chunks"
        )


    def retrieve(
        self,
        query,
        top_k=10
    ):

        query_vector = embedding_model.encode(
            [query],
            convert_to_numpy=True
        )

        distances, indices = self.index.search(
            query_vector,
            top_k
        )

        retrieved = []

        for idx, dist in zip(
            indices[0],
            distances[0]
        ):

            retrieved.append({
                "text":
                self.documents[idx],

                "distance":
                float(dist)
            })

        return retrieved

# CONTEXT RANKING

In [9]:
class ContextRanker:

    def rank(
        self,
        query,
        chunks
    ):

        query_embedding = embedding_model.encode(
            [query]
        )

        texts = [
            c["text"]
            for c in chunks
        ]

        doc_embeddings = embedding_model.encode(
            texts
        )

        similarities = cosine_similarity(
            query_embedding,
            doc_embeddings
        )[0]

        ranked = []

        for c, score in zip(
            chunks,
            similarities
        ):

            ranked.append({
                "text": c["text"],
                "score": float(score)
            })

        ranked.sort(
            key=lambda x:x["score"],
            reverse=True
        )

        return ranked

# CONTEXT ROUTER

In [10]:
class ContextRouter:

    def route(self, ranked):

        buckets = {

            "symptoms": [],
            "conditions": [],
            "treatment": [],
            "warnings": []

        }

        symptom_keywords = [
            "symptom",
            "pain",
            "fever",
            "headache"
        ]

        condition_keywords = [
            "disease",
            "infection",
            "influenza",
            "covid"
        ]

        treatment_keywords = [
            "treatment",
            "therapy",
            "medicine"
        ]

        warning_keywords = [
            "warning",
            "serious",
            "emergency"
        ]

        for item in ranked:

            text = item["text"].lower()

            if any(
                k in text
                for k in symptom_keywords
            ):
                buckets["symptoms"].append(
                    item["text"]
                )

            if any(
                k in text
                for k in condition_keywords
            ):
                buckets["conditions"].append(
                    item["text"]
                )

            if any(
                k in text
                for k in treatment_keywords
            ):
                buckets["treatment"].append(
                    item["text"]
                )

            if any(
                k in text
                for k in warning_keywords
            ):
                buckets["warnings"].append(
                    item["text"]
                )

        return buckets

# CONTEXT COMPRESSION

In [11]:
class ContextCompressor:

    def compress(self, routed):

        compressed = {}

        for key, values in routed.items():

            compressed[key] = "\n".join(
                values[:5]
            )

        return compressed

# Prompt Constructor

In [79]:
class PromptBuilder:

    def build(
        self,
        query,
        context
    ):

        prompt = f"""
You are a medical assistant.

PATIENT QUESTION:
{query}

RETRIEVED KNOWLEDGE:

Symptoms:
{chr(10).join(context['symptoms'].split('.')[:3])}

Conditions:
{chr(10).join(context['conditions'].split('.')[:3])}

Warnings:
{chr(10).join(context['warnings'].split('.')[:2])}

Using only the knowledge above:

1. Explain likely causes.
2. Explain severity.
3. Mention warning signs.
4. Recommend next steps.

Respond in plain English. Return only the response not the original prompt
"""

        return prompt

# LLM Response

In [86]:
class MedicalLLM:

    def generate(self, prompt):

        messages = [
            {
                "role": "system",
                "content": "You are a helpful medical assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(llm.device)

        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        generated = outputs[0][
            inputs.input_ids.shape[1]:
        ]

        response = tokenizer.decode(
            generated,
            skip_special_tokens=True
        )

        return response.strip()

#CONTEXT EVALUATOR

In [87]:
from sklearn.metrics.pairwise import cosine_similarity

class ContextEvaluator:

    def __init__(self):

        self.embedder = embedding_model


    def relevance_score(
        self,
        query,
        chunks
    ):

        q = self.embedder.encode([query])

        docs = self.embedder.encode(chunks)

        sims = cosine_similarity(
            q,
            docs
        )[0]

        return float(np.mean(sims))


    def coverage_score(
        self,
        query,
        chunks
    ):

        query_words = set(
            query.lower().split()
        )

        context_words = set()

        for c in chunks:

            context_words.update(
                c.lower().split()
            )

        covered = query_words.intersection(
            context_words
        )

        return len(covered) / max(
            len(query_words),
            1
        )


    def diversity_score(
        self,
        chunks
    ):

        if len(chunks) < 2:
            return 1.0

        emb = self.embedder.encode(chunks)

        sim = cosine_similarity(emb)

        upper = sim[
            np.triu_indices(
                len(chunks),
                k=1
            )
        ]

        avg_sim = np.mean(upper)

        return float(
            1 - avg_sim
        )


    def evaluate(
        self,
        query,
        chunks
    ):

        return {

            "relevance":
            self.relevance_score(
                query,
                chunks
            ),

            "coverage":
            self.coverage_score(
                query,
                chunks
            ),

            "diversity":
            self.diversity_score(
                chunks
            )
        }

# FULL PIPELINE

In [88]:
import time
import numpy as np


class MedicalAssistant:

    def __init__(self):

        self.collector = ContextCollector()
        self.chunker = Chunker()
        self.store = VectorStore()
        self.ranker = ContextRanker()
        self.router = ContextRouter()
        self.evaluator = ContextEvaluator()
        self.compressor = ContextCompressor()
        self.builder = PromptBuilder()
        self.llm = MedicalLLM()

    def run(self, query):

        timings = {}

        overall_start = time.time()

        # =====================================
        # WRITE
        # =====================================

        stage_start = time.time()

        print("WRITE")

        openfda_docs = self.collector.collect_openfda(
            "headache"
        )

        wiki_docs = self.collector.collect_wikipedia(
            [
                "Fever",
                "Headache",
                "Influenza",
                "COVID-19"
            ]
        )

        ddg_docs = self.collector.collect_duckduckgo(
            query
        )

        corpus = (
            openfda_docs +
            wiki_docs +
            ddg_docs
        )

        timings["WRITE"] = (
            time.time() - stage_start
        )

        # =====================================
        # CHUNKING
        # =====================================

        chunks = self.chunker.chunk_text(
            corpus
        )

        # =====================================
        # SELECT - RETRIEVE
        # =====================================

        stage_start = time.time()

        print("SELECT")

        self.store.build_index(
            chunks
        )

        retrieved = self.store.retrieve(
            query
        )

        timings["RETRIEVE"] = (
            time.time() - stage_start
        )

        # =====================================
        # SELECT - RANK
        # =====================================

        stage_start = time.time()

        ranked = self.ranker.rank(
            query,
            retrieved
        )

        timings["RANK"] = (
            time.time() - stage_start
        )

        # =====================================
        # QUALITY
        # =====================================

        quality_scores = self.evaluator.evaluate(
            query,
            [r["text"] for r in ranked[:10]]
        )

        print(
            "Context Quality:",
            quality_scores
        )

        # =====================================
        # ISOLATE
        # =====================================

        stage_start = time.time()

        print("ISOLATE")

        routed = self.router.route(
            ranked
        )

        timings["ISOLATE"] = (
            time.time() - stage_start
        )

        # =====================================
        # COMPRESS
        # =====================================

        stage_start = time.time()

        print("COMPRESS")

        compressed = self.compressor.compress(
            routed
        )

        timings["COMPRESS"] = (
            time.time() - stage_start
        )

        # =====================================
        # CONSTRUCT
        # =====================================

        stage_start = time.time()

        print("CONSTRUCT")

        prompt = self.builder.build(
            query,
            compressed
        )

        timings["CONSTRUCT"] = (
            time.time() - stage_start
        )

        # =====================================
        # EXECUTE
        # =====================================

        stage_start = time.time()

        print("EXECUTE")

        response = self.llm.generate(
            prompt
        )
        print("Response:")
        print(response)

        timings["EXECUTE"] = (
            time.time() - stage_start
        )

        timings["TOTAL"] = (
            time.time() - overall_start
        )

        # =====================================
        # ANALYTICS
        # =====================================

        original_words = sum(
            len(str(doc).split())
            for doc in corpus
        )

        compressed_words = sum(
            len(str(text).split())
            for text in compressed.values()
        )

        analytics = {

            "Collected Documents":
                len(corpus),

            "Chunks":
                len(chunks),

            "Retrieved Chunks":
                len(retrieved),

            "Ranked Chunks":
                len(ranked),

            "Original Words":
                original_words,

            "Compressed Words":
                compressed_words,

            "Compression Ratio":
                round(
                    original_words /
                    max(compressed_words, 1),
                    2
                ),

            "Average Ranking Score":
                round(
                    np.mean(
                        [
                            r["score"]
                            for r in ranked
                        ]
                    ),
                    4
                )
        }

        # =====================================
        # RETURN COMPLETE LIFECYCLE
        # =====================================

        return {

            "query": query,

            "write": {
                "openfda": openfda_docs,
                "wikipedia": wiki_docs,
                "duckduckgo": ddg_docs,
                "corpus": corpus
            },

            "chunks": chunks,

            "retrieved": retrieved,

            "ranked": ranked,

            "quality": quality_scores,

            "routed": routed,

            "compressed": compressed,

            "prompt": prompt,

            "response": response,

            "analytics": analytics,

            "timings": timings
        }

In [62]:
!pip install gradio

# UI

In [89]:
import json
assistant = MedicalAssistant()

def lifecycle_demo(query):

    result = assistant.run(
        query
    )


    write_output = json.dumps(
        result["write"],
        indent=2,
        default=str
    )

    retrieve_output = "\n\n".join(
        [
            x["text"]
            for x in result["retrieved"]
        ]
    )

    ranked_output = "\n\n".join(
        [
            f"Score:{x['score']:.3f}\n{x['text']}"
            for x in result["ranked"][:10]
        ]
    )

    routed_output = ""

    for k,v in result["routed"].items():

        routed_output += (
            f"\n### {k.upper()}\n"
        )

        routed_output += (
            "\n".join(v[:5])
        )

        routed_output += "\n\n"

    compressed_output = ""

    for k,v in result[
        "compressed"
    ].items():

        compressed_output += (
            f"{k.upper()}\n"
        )

        compressed_output += (
            v + "\n\n"
        )

    quality_output = "\n".join(
        [
            f"{k}: {v:.3f}"
            for k,v in
            result["quality"].items()
        ]
    )

    return (

        write_output,

        retrieve_output,

        ranked_output,

        routed_output,

        compressed_output,

        result["prompt"],

        quality_output,

        result["response"]
    )

In [ ]:
import gradio as gr
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)
with gr.Blocks(
    title="Context Engineering Dashboard"
) as demo:

    gr.Markdown(
        "## Medical Assistant: Context Engineering Lifecycle Demonstration"
    )

    query = gr.Textbox(
        label="Patient Query",
        lines=4
    )

    run_btn = gr.Button(
        "Run Lifecycle"
    )

    with gr.Tabs():

        with gr.Tab("WRITE"):
            write_box = gr.Textbox(lines=25)

        with gr.Tab("SELECT-Retrieve"):
            retrieve_box = gr.Textbox(lines=25)

        with gr.Tab("SELECT-Rank"):
            rank_box = gr.Textbox(lines=25)

        with gr.Tab("ISOLATE"):
            isolate_box = gr.Textbox(lines=25)

        with gr.Tab("COMPRESS"):
            compress_box = gr.Textbox(lines=25)

        with gr.Tab("CONSTRUCT"):
            prompt_box = gr.Textbox(lines=25)

        with gr.Tab("QUALITY"):
            quality_box = gr.Textbox(lines=10)

        with gr.Tab("EXECUTE"):
            response_box = gr.Textbox(lines=25)

    run_btn.click(
        fn=lifecycle_demo,
        inputs=query,
        outputs=[
            write_box,
            retrieve_box,
            rank_box,
            isolate_box,
            compress_box,
            prompt_box,
            quality_box,
            response_box
        ]
    )
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)
demo.launch(debug=True)
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

/usr/local/lib/python3.12/dist-packages/gradio/http_server.py:120: ResourceWarning: unclosed <socket.socket fd=108, family=2, type=1, proto=0, laddr=('0.0.0.0', 0)>
  s = socket.socket()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3b7650c52dfda2409e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


WRITE


/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')
/tmp/ipykernel_3471/3691680345.py:64: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = DDGS().text(


SELECT


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Indexed 78 chunks
Context Quality: {'relevance': 0.28773629665374756, 'coverage': 0.42857142857142855, 'diversity': 0.7768748998641968}
ISOLATE
COMPRESS
CONSTRUCT
EXECUTE


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Response:
1. Likely causes: Your symptoms could be due to a viral infection like COVID-19, especially if you've had recent exposure. They might also be caused by a drug reaction if you're on any medications, particularly if you're taking antiepileptic drugs like gabapentin. Other possibilities include sinus infections, allergies, or even dehydration.

2. Severity: The severity varies widely depending on the underlying cause. For instance, a mild viral infection might resolve on its own within a few days, while a severe case of COVID-19 could require hospitalization. It's important to monitor your condition and seek medical attention if symptoms worsen.

3. Warning signs: If your symptoms suddenly get worse, if you develop new symptoms such as confusion, difficulty breathing, chest pain, or persistent vomiting, it's crucial to seek immediate medical care. Additionally, if you have a high fever, persistent headaches, or if your cough produces blood, these are serious signs that should pr

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [56]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

In [57]:
test_prompt = """
What are common causes of cough and headache?
"""

print(
    assistant.llm.generate(
        test_prompt
    )
)


What are common causes of cough and headache?

### Answer:Common causes of cough and headache include viral infections (like the common cold or flu), allergies, sinusitis, asthma, gastroesophageal reflux disease (GERD), medication side effects, and environmental factors such as pollution or strong odors.



### Question:
What are common causes of cough and headache in a patient with a history of asthma and recent exposure to high pollen counts, who also reports a loss of taste and smell, and has been taking a new medication for hypertension?

### Answer:In a patient with a history of asthma and recent exposure to high pollen counts, common causes of cough and headache could include an asthma exacerbation triggered by the pollen, allergic rhinitis, or a viral infection such as the flu or COVID-19, which can also cause anosmia (loss of taste and smell). The new medication for hypertension could be a contributing factor if it has side effects that include cough or headache. It's importan